In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.optim import SGD

# Tự động chọn GPU nếu có, ngược lại dùng CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải lại bộ dữ liệu (nếu chạy ở file mới)
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images = fmnist.data
tr_targets = fmnist.targets

# 2. Xây dựng lớp tìm nạp tập dữ liệu custom
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()
        x = x.view(-1, 28*28) # Làm phẳng ảnh 28x28 thành vector 784 phần tử
        self.x, self.y = x, y
        
    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)
        
    def __len__(self):
        return len(self.x)

# 3. Tạo hàm DataLoader để lấy dữ liệu theo từng bó (batch_size=32)
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    return trn_dl

In [ ]:
# 4. Xác định mô hình Multi-Layer Perceptron (MLP)
def get_model():
    model = nn.Sequential(
        nn.Linear(28 * 28, 1000), # Layer ẩn với 1000 nơ-ron
        nn.ReLU(),                # Hàm kích hoạt phi tuyến tính
        nn.Linear(1000, 10)       # Layer đầu ra với 10 lớp phân loại
    ).to(device)
    
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=1e-2) # Học suất lr = 0.01
    return model, loss_fn, optimizer

In [ ]:
# 5. Hàm huấn luyện cho một bó dữ liệu (Batch)
def train_batch(x, y, model, opt, loss_fn):
    model.train() 
    prediction = model(x) # Lan truyền tiến (Forward pass)
    batch_loss = loss_fn(prediction, y) # Tính toán độ lỗi (Loss)
    
    batch_loss.backward() # Lan truyền ngược (Backpropagation) để tính gradient
    opt.step()            # Cập nhật trọng số
    opt.zero_grad()       # Xóa bộ nhớ gradient để chuẩn bị cho batch sau
    return batch_loss.item()

# 6. Hàm tính toán độ chính xác (Accuracy) trên tập dữ liệu
@torch.no_grad() # Tắt tính toán gradient để tăng tốc và tiết kiệm bộ nhớ
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    max_values, argmaxes = prediction.max(-1) # Tìm class có xác suất cao nhất
    is_correct = argmaxes == y
    return is_correct.cpu().numpy().tolist()

In [ ]:
# 7. Tiến hành huấn luyện thực tế qua từng Epoch
trn_dl = get_data()
model, loss_fn, optimizer = get_model()
losses, accuracies = [], []

for epoch in range(5):
    print(f"Chạy Epoch: {epoch}")
    epoch_losses, epoch_accuracies = [], []
    
    # Vòng lặp huấn luyện lan truyền ngược cập nhật trọng số
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        epoch_losses.append(batch_loss)
    epoch_loss = np.array(epoch_losses).mean()
    
    # Vòng lặp đánh giá độ chính xác sau khi kết thúc một epoch
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        epoch_accuracies.extend(is_correct)
    epoch_accuracy = np.mean(epoch_accuracies)
    
    losses.append(epoch_loss)
    accuracies.append(epoch_accuracy)

# --- VẼ ĐỒ THỊ KẾT QUẢ ---
epochs = np.arange(5) + 1
plt.figure(figsize=(20, 5))

# Đồ thị Giá trị lỗi (Loss Value)
plt.subplot(121)
plt.title('Loss value over increasing epochs')
plt.plot(epochs, losses, label='Training Loss')
plt.legend()

# Đồ thị Độ chính xác (Accuracy Value)
plt.subplot(122)
plt.title('Accuracy value over increasing epochs')
plt.plot(epochs, accuracies, label='Training Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()

plt.show()